In [1]:
import requests
import pandas as pd
import os

## Étape 1 — Extraction / Bronze
Gérer les erreurs lors des appels API : timeout, erreurs HTTP, réponses invalides, etc.

1. Récupérer le dataset des villes marocaines.

In [7]:
ma_url = "https://simplemaps.com/data/ma-cities"
ma_json_url = "https://simplemaps.com/static/data/country-cities/ma/ma.json"

try:
    response = requests.get(url=ma_json_url, timeout=60)
    response.raise_for_status()
    ma_df = pd.DataFrame(data=response.json() or [])
    ma_df.head(10)
except requests.exceptions.Timeout:
    print("L'API a pris plus d'une minute pour répondre.")
except requests.exceptions.HTTPError as e:
    print(f"Erreur HTTP: {e}")
except requests.exceptions.JSONDecodeError:
    print("Format json invalide.")
# La base des exceptions sourvenues lors d'un request.
except requests.exceptions.RequestException as r:
    print(f"Request Exception: {r}")

2. Utiliser les coordonnées des villes pour interroger l'API Open-Meteo.

In [8]:
meteo_data = []
meteo_url = "https://api.open-meteo.com/v1/forecast"

for lat, lng in ma_df[['lat', 'lng']].to_numpy():
    meteo_params = {
        "latitude": lat,
        "longitude": lng,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "current": "weather_code",
        "forecast_days": 3
    }
    try:
        response = requests.get(url=meteo_url, params=meteo_params, timeout=300)
        response.raise_for_status()
        meteo_data.append(response.json())
    except requests.exceptions.Timeout:
        print("L'API a pris plus de cinqs minutes pour répondre.")
    except requests.exceptions.HTTPError as e:
        print(f"Erreur HTTP: {e}")
    except requests.exceptions.JSONDecodeError:
        print("Format json invalide.")
    except requests.exceptions.RequestException as r:
        print(f"Request Exception: {r}")

Erreur HTTP: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=33.5992&longitude=-7.6200&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=precipitation_probability_max&daily=wind_speed_10m_max&daily=wind_gusts_10m_max&current=weather_code&forecast_days=3
Erreur HTTP: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=35.7767&longitude=-5.8039&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=precipitation_probability_max&daily=wind_speed_10m_max&daily=wind_gusts_10m_max&current=weather_code&forecast_days=3
Erreur HTTP: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=34.0433&longitude=-5.0033&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=precipitation_probability_max&daily=wind_speed_10m_max&daily=wind_gusts_10m_max&current=weather_code&forecast_days=3
Erreur HTTP: 

KeyboardInterrupt: 

3. Récupérer les prévisions météorologiques quotidiennes des prochains jours.

In [39]:
meteo_df = pd.DataFrame(data=meteo_data)
meteo_df

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,current_units,current,daily_units,daily
0,33.56250,-7.625000,0.150561,0,GMT,GMT,23.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
1,35.75000,-5.812500,0.097156,0,GMT,GMT,30.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
2,34.06250,-5.000000,0.118375,0,GMT,GMT,389.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
3,31.62500,-8.000000,0.193834,0,GMT,GMT,469.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
4,34.00000,-6.812500,0.100493,0,GMT,GMT,27.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
...,...,...,...,...,...,...,...,...,...,...,...
115,32.43750,-6.312500,0.178456,0,GMT,GMT,503.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
116,32.68750,-5.937500,0.138164,0,GMT,GMT,657.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
117,31.18750,-8.875000,0.305891,0,GMT,GMT,863.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."
118,32.12500,-9.062500,0.133753,0,GMT,GMT,173.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-14T11:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-14', '2026-09-15', '2026-09..."


4. Conserver les données brutes dans bronze/.

In [40]:
try:
    os.mkdir(path="./bronze")
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except FileExistsError:
    if "meteo.csv" not in os.listdir("./bronze"):
        meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except Exception as e:
    print(e)

## Étape 2 — Nettoyage / Silver

1. standardiser les types et les dates.

2. détecter les doublons et incohérences.

3. contrôler la qualité des données.

4. effectuer la jointure entre les villes et les données météo.

5. stocker les données nettoyées dans silver/.

## Étape 3 — Feature Engineering / Gold

1. catégories de température.

2. catégories de précipitations.

3. catégories de vent.

4. date.

5. autres indicateurs pertinents.

6. Weather Risk Score
<br>
Créer un score de risque météorologique de 0 à 100 permettant d'identifier les conditions potentiellement défavorables.

7. Vous devrez justifier :
* les variables utilisées.
* les seuils.
* la méthode de calcul.

8. Charger les données finales dans PostgreSQL.<br>
Le modèle devra permettre de gérer au minimum :
* les villes et leurs coordonnées.
* les prévisions météorologiques.
* le risk_score.<br>
Vous devrez également prévoir une stratégie pour éviter les doublons lors des nouvelles exécutions du pipeline, car les prévisions peuvent être mises à jour.

9. Bonus:
<br>
conserver l'historique des différentes prévisions.

## Étape 4 — Analyse SQL
Réaliser au minimum 5 requêtes SQL répondant à des questions métier.

1. Quelles villes auront les températures les plus élevées ?

2. Quelles villes auront les plus fortes précipitations ?

3. Quelles villes présentent le risque moyen le plus élevé ?

4. Quelles périodes présentent le risque maximal ?

5. Pour chaque ville, quelle période présente le plus grand risque ?

6. Bonus:
sous-requêtes, fonctions de fenêtrage.

## Étape 5 — Dashboard Streamlit
Créer un dashboard connecté à PostgreSQL permettant de visualiser les prévisions et les risques.
<br>
Le dashboard doit permettre de répondre rapidement à la question :
Où et quand faut-il être particulièrement vigilant dans les prochains jours ?

1. KPI (Key Performance Indicator):
* nombre de villes.
* température maximale.
* précipitations maximales.
* nombre de périodes à risque.
* ville présentant le risque le plus élevé.

2. Filtres:
<br>
Permettre de filtrer notamment par : 
* ville. 
* date. 
* période. 
* niveau de risque.

## Étape 6 — Orchestration & automatisation (Airflow)

1. Définir un
DAG Airflow
qui automatise l'ensemble du pipeline :
* Extraction depuis l'API.
* Nettoyage / transformation.
* Feature engineering & chargement dans PostgreSQL.
* (Rafraîchissement des données pour le dashboard).

2. Planifier une exécution automatique (ex. quotidienne) et gérer les échecs (retries).

3. Conteneuriser le projet avec Docker Compose (Postgres + Airflow + Streamlit).

4. Bonus:
* Ajouter du logging structuré et des alertes en cas d'échec du DAG.
* Historique des prévisions.
* Pipeline incrémental.
* Contrôles de qualité.